# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to explore and process the FAIR² dataset using the `mlcroissant` library. The dataset is described by a [Croissant schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) and contains multiple record sets, fields, and columns accessible by their `@id`s, following standards for reproducible data science workflows.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. This reveals the dataset structure and documentation, making it easy to reference its contents by their `@id` fields.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Print basic metadata information
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their `@id` references. All further references to fields, record sets, or columns will use their `@id` values to ensure consistent access and reproducibility.

**List of record sets and their fields:**

In [ ]:
# Display all record sets and their fields using their @id references
from pprint import pprint

record_sets = list(dataset.record_sets)
print(f"Record sets found ({len(record_sets)}):\n")
for rs in record_sets:
    print(f"Record set: {rs['@id']}")
    rs_fields = rs.get('field', [])
    if isinstance(rs_fields, dict):
        rs_fields = [rs_fields]
    elif not isinstance(rs_fields, list):
        rs_fields = []
    print("  Fields:")
    for field in rs_fields:
        if isinstance(field, dict):
            print(f"    - {field.get('@id','<unknown>')} (Data type: {field.get('dataType','n/a')})")
        else:
            print(f"    - {field}")
    print()

Let's look at a few sample records from the principal record set. Pick a record set `@id` from the above output (e.g., for the main clinical tabular data).

In [ ]:
# List the @ids for all available record sets for human selection
principal_rs_id = None
if len(record_sets) >= 1:
    # For this dataset typically the principal tabular data is the first record set
    principal_rs_id = record_sets[0]['@id']
    print(f"Principal record set @id: {principal_rs_id}")
else:
    print("No record sets found in the dataset.")

In [ ]:
# Preview the first 3 records in the principal record set by @id
if principal_rs_id:
    print(f"First 3 records from record set '{principal_rs_id}':\n")
    for i, record in enumerate(dataset.records(record_set=principal_rs_id)):
        print(record)
        if i >= 2:
            break

## 3. Data Extraction
Load all records from each record set into Pandas DataFrames for further analysis. All references to record sets and fields use their `@id` fields for robustness and reproducibility.

In [ ]:
# Extract all record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]
# Load records for each record set into a DataFrame
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records from record set {rs_id}")
    else:
        print(f"No records found for record set {rs_id}")

# Print preview of the columns of the main record set
main_rs_id = principal_rs_id
if main_rs_id in dataframes:
    print("\nMain record set columns:")
    print(dataframes[main_rs_id].columns.tolist())
    # Show first few rows
    dataframes[main_rs_id].head()
else:
    print(f"No data loaded for main record set {main_rs_id}.")

## 4. Exploratory Data Analysis (EDA)
Apply typical data processing steps. All field names are referenced using the field `@id`. Examples shown here include:
  - Filtering numeric values
  - Normalization
  - Grouping and statistics

You may replace the selected field `@id` values with your own from the DataFrame columns above.

In [ ]:
# Choose a numeric field `@id` for demonstration.
# Replace this with the target field's actual @id from the printout above as appropriate for your analysis.
numeric_field_id = None
group_field_id = None
df = dataframes.get(main_rs_id)

if df is not None:
    # Heuristic: try to auto-select a likely numeric field @id
    numeric_candidates = [col for col in df.columns if (df[col].dropna().apply(lambda x: isinstance(x, (int, float, np.number)) or (isinstance(x, str) and x.replace('.','',1).isdigit())).all())]
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
        print(f"Auto-selected numeric field: {numeric_field_id}")
    # Heuristic: group by a likely categorical field @id
    non_numeric = [col for col in df.columns if col != numeric_field_id]
    if non_numeric:
        group_field_id = non_numeric[0]
        print(f"Auto-selected group field: {group_field_id}")

    # Attempt to coerce numeric if substrate is a string
    if numeric_field_id:
        # Convert to numeric, coerce errors, then filter
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = df[numeric_field_id].mean()
        print(f"Filtering {numeric_field_id} for values > {threshold:.2f}\n")
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (showing up to 5):")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        if group_field_id and group_field_id in filtered_df.columns:
            print(f"Grouped mean of {numeric_field_id} by {group_field_id} (first few groups):")
            grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            display(grouped.head())
else:
    print("No data was loaded for the selected record set.")

## 5. Visualization
Visualize the distribution of the selected numeric field and/or relationships between fields.

**Note:** Plots below use field `@id` as axis labels to ensure transparency and reproducibility.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()
    
    # Boxplot grouped by group_field_id (if available)
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to:
- Load and inspect a Croissant-described dataset using `mlcroissant`
- Reference all data entities (record sets, fields) by their `@id` values for clarity and reproducibility
- Extract, normalize, and visualize tabular data

You can now repeat these steps for other record sets or fields of interest using their `@id`s for robust, future-proof data exploration and analysis.